# 🏥 MediSpa Bay Area — Financial Forecast
**P&L · Cash Flow · ROI | 36-Month Projection**

Adjust the assumptions in **Section 1** to match your business. Everything else recalculates automatically.

In [ ]:
# ── Install dependencies if needed ────────────────────────────────────────────
# Uncomment if running for the first time
# !pip install pandas numpy matplotlib ipywidgets --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

# Notebook display settings
pd.set_option('display.float_format', '${:,.0f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})
print('✅ Libraries loaded')

---
## 1. 🎛️ Assumptions — Edit These
All inputs are centralized here. Modify and re-run cells below.

In [ ]:
# ── FORECAST HORIZON ──────────────────────────────────────────────────────────
MONTHS = 36

# ── SERVICES: {name: (avg_price, sessions/month, annual_growth_rate)} ─────────
# annual_growth_rate: e.g. 0.06 = 6% per year, compounded monthly
SERVICES = {
    #  Service Name          price   sessions  annual_growth
    "Botox / Fillers":      (650,    120,       0.05),
    "Laser Treatments":     (400,     80,       0.06),
    "Facials / Skincare":   (180,    200,       0.04),
    "Body Contouring":      (900,     40,       0.07),
    "IV Therapy":           (250,     60,       0.08),
    "Retail Products":      (120,    150,       0.03),
}

# ── COGS as % of revenue per service ─────────────────────────────────────────
COGS_PCT = {
    "Botox / Fillers":     0.30,   # injectables are expensive
    "Laser Treatments":    0.15,   # high margin after equipment payoff
    "Facials / Skincare":  0.20,
    "Body Contouring":     0.12,
    "IV Therapy":          0.35,   # supplies add up
    "Retail Products":     0.50,   # standard retail margin
}

# ── MONTHLY FIXED OPERATING EXPENSES (Bay Area benchmarks) ───────────────────
FIXED_OPEX = {
    "Rent (NNN)": 12_000,
    "Staff Salaries": 45_000,
    "Medical Director": 8_000,
    "Marketing / SEO": 5_000,
    "Software / EMR": 800,
    "Insurance (med mal)": 2_500,
    "Utilities": 1_800,
    "Supplies (non-COGS)": 1_500,
    "Misc / Admin": 1_200,
}

VARIABLE_OPEX_PCT = 0.04   # % of revenue (credit card fees, consumables)

# ── INITIAL INVESTMENT ────────────────────────────────────────────────────────
INITIAL_INVESTMENT = {
    "Equipment": 180_000,
    "Leasehold Improvements": 120_000,
    "Licenses / Legal": 15_000,
    "Working Capital": 80_000,
    "Marketing Launch": 25_000,
}

# ── CASH FLOW & TAX SETTINGS ──────────────────────────────────────────────────
RECEIVABLES_DAYS = 5     # credit card settlement lag
PAYABLES_DAYS = 30       # pay suppliers net-30
TAX_RATE = 0.28          # blended CA state + federal

# ── DERIVED CONSTANTS ─────────────────────────────────────────────────────────
TOTAL_INVESTMENT = sum(INITIAL_INVESTMENT.values())
MONTHLY_DA = (
    INITIAL_INVESTMENT["Equipment"] / (5 * 12) +
    INITIAL_INVESTMENT["Leasehold Improvements"] / (10 * 12)
)

print(f'✅ Assumptions set | Total Investment: ${TOTAL_INVESTMENT:,.0f} | Monthly D&A: ${MONTHLY_DA:,.0f}')

---
## 2. 🔧 Forecast Engine

In [ ]:
def build_forecast(services=SERVICES, fixed_opex=FIXED_OPEX, cogs_pct=COGS_PCT,
                   variable_opex_pct=VARIABLE_OPEX_PCT, total_investment=TOTAL_INVESTMENT,
                   monthly_da=MONTHLY_DA, months=MONTHS):
    """
    Build a monthly P&L, Cash Flow, and ROI forecast.
    Returns a tidy DataFrame with one row per month.
    """
    rows = []
    cum_net_income = 0
    cum_cash = -total_investment  # starts negative (we spent money)
    prev_revenue = prev_cogs = 0

    for m in range(1, months + 1):

        # ── Revenue ───────────────────────────────────────────────────────────
        service_rev = {
            svc: price * sessions * (1 + growth / 12) ** (m - 1)
            for svc, (price, sessions, growth) in services.items()
        }
        total_revenue = sum(service_rev.values())

        # ── COGS & Gross Profit ───────────────────────────────────────────────
        total_cogs = sum(service_rev[svc] * cogs_pct[svc] for svc in service_rev)
        gross_profit = total_revenue - total_cogs

        # ── OpEx & EBITDA ─────────────────────────────────────────────────────
        fixed = sum(fixed_opex.values())
        variable = total_revenue * variable_opex_pct
        ebitda = gross_profit - fixed - variable

        # ── Net Income ────────────────────────────────────────────────────────
        ebit = ebitda - monthly_da
        tax = max(ebit * TAX_RATE, 0)
        net_income = ebit - tax

        # ── Cash Flow from Operations ─────────────────────────────────────────
        # Working capital changes (AR goes up → uses cash; AP goes up → frees cash)
        delta_ar = (total_revenue - prev_revenue) * (RECEIVABLES_DAYS / 30)
        delta_ap = (total_cogs - prev_cogs) * (PAYABLES_DAYS / 30)
        cfo = net_income + monthly_da - delta_ar + delta_ap

        # ── Cumulative Metrics ────────────────────────────────────────────────
        cum_net_income += net_income
        cum_cash += cfo
        roi = (cum_net_income / total_investment) * 100

        rows.append({
            "Month": m,
            # P&L
            "Revenue": total_revenue,
            "COGS": total_cogs,
            "Gross_Profit": gross_profit,
            "Gross_Margin_%": gross_profit / total_revenue * 100,
            "Fixed_OpEx": fixed,
            "Variable_OpEx": variable,
            "EBITDA": ebitda,
            "EBITDA_Margin_%": ebitda / total_revenue * 100,
            "DA": monthly_da,
            "EBIT": ebit,
            "Tax": tax,
            "Net_Income": net_income,
            "Net_Margin_%": net_income / total_revenue * 100,
            # Cash Flow
            "CFO": cfo,
            "Cum_Cash_Flow": cum_cash,
            # ROI
            "Cum_Net_Income": cum_net_income,
            "ROI_%": roi,
        })

        prev_revenue, prev_cogs = total_revenue, total_cogs

    return pd.DataFrame(rows).set_index("Month")


df_base = build_forecast()
print(f'✅ Base forecast built ({MONTHS} months)')
df_base.head(3)

---
## 3. 📊 Scenario Analysis — Bear / Base / Bull

In [ ]:
import copy

def scale_services(services, volume_factor=1.0, price_factor=1.0):
    """Return a new services dict with volume and/or price scaled."""
    return {
        svc: (price * price_factor, int(sessions * volume_factor), growth)
        for svc, (price, sessions, growth) in services.items()
    }

# Bear: 30% lower volume, higher rent
bear_opex = {**FIXED_OPEX, "Rent (NNN)": 16_000}
bear_services = scale_services(SERVICES, volume_factor=0.70)

# Bull: 25% higher volume, 10% price increase
bull_services = scale_services(SERVICES, volume_factor=1.25, price_factor=1.10)

scenarios = {
    "🐻 Bear": build_forecast(services=bear_services, fixed_opex=bear_opex),
    "📊 Base": build_forecast(),
    "🐂 Bull": build_forecast(services=bull_services),
}

print('✅ Scenarios built')
for name, df in scenarios.items():
    be = df[df["Cum_Cash_Flow"] >= 0].index.min()
    print(f"  {name:12s} | Month 36 Revenue: ${df.iloc[-1]['Revenue']:>10,.0f} "
          f"| Net Income: ${df.iloc[-1]['Net_Income']:>9,.0f} "
          f"| Cash Breakeven: Month {be if pd.notna(be) else '>36'}")

---
## 4. 📈 Dashboard Charts

In [ ]:
COLORS = {"🐻 Bear": "#E53935", "📊 Base": "#1E88E5", "🐂 Bull": "#43A047"}
months_axis = range(1, MONTHS + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("MediSpa Bay Area — 36-Month Financial Forecast", fontsize=14, fontweight="bold", y=1.01)

plots = [
    (axes[0,0], "Revenue",       "Monthly Revenue ($K)",           "$K"),
    (axes[0,1], "Net_Income",    "Monthly Net Income ($K)",        "$K"),
    (axes[1,0], "Cum_Cash_Flow", "Cumulative Cash Flow ($K)",      "$K"),
    (axes[1,1], "ROI_%",         "Cumulative ROI (%)",             "%"),
]

for ax, col, title, ylabel in plots:
    for name, df in scenarios.items():
        vals = df[col] / (1000 if ylabel == "$K" else 1)
        ax.plot(months_axis, vals, label=name, color=COLORS[name], linewidth=2)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    if ylabel == "$K":
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:.0f}K"))
    else:
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))

plt.tight_layout()
plt.savefig("medispa_dashboard.png", dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → medispa_dashboard.png')

---
## 5. 📋 P&L Statement — Annual View

In [ ]:
def annual_pnl(df, label="Base"):
    """Collapse monthly forecast into annual P&L table."""
    df = df.reset_index()
    df["Year"] = ((df["Month"] - 1) // 12) + 1
    annual = df.groupby("Year").agg(
        Revenue=("Revenue", "sum"),
        COGS=("COGS", "sum"),
        Gross_Profit=("Gross_Profit", "sum"),
        Total_OpEx=("Fixed_OpEx", "sum"),
        EBITDA=("EBITDA", "sum"),
        DA=("DA", "sum"),
        EBIT=("EBIT", "sum"),
        Tax=("Tax", "sum"),
        Net_Income=("Net_Income", "sum"),
        Avg_Gross_Margin=("Gross_Margin_%", "mean"),
        Avg_EBITDA_Margin=("EBITDA_Margin_%", "mean"),
        Avg_Net_Margin=("Net_Margin_%", "mean"),
    ).round(0)
    annual.index = [f"Year {y}" for y in annual.index]
    annual.columns.name = label
    return annual

print("=" * 70)
print("P&L — BASE CASE (Annual)")
print("=" * 70)

pnl = annual_pnl(df_base)

# Format for display
dollar_cols = ["Revenue","COGS","Gross_Profit","Total_OpEx","EBITDA","DA","EBIT","Tax","Net_Income"]
pct_cols    = ["Avg_Gross_Margin","Avg_EBITDA_Margin","Avg_Net_Margin"]

display_pnl = pnl.copy()
for c in dollar_cols:
    display_pnl[c] = display_pnl[c].apply(lambda x: f"${x:>12,.0f}")
for c in pct_cols:
    display_pnl[c] = display_pnl[c].apply(lambda x: f"{x:>8.1f}%")

display_pnl.T

---
## 6. 💵 Cash Flow Statement — Annual View

In [ ]:
def annual_cashflow(df):
    df = df.reset_index()
    df["Year"] = ((df["Month"] - 1) // 12) + 1
    cf = df.groupby("Year").agg(
        Net_Income=("Net_Income", "sum"),
        DA_Addback=("DA", "sum"),
        CFO=("CFO", "sum"),
    ).round(0)
    # Capex only in Year 0 (pre-launch); show as reference
    cf["CFI (Capex)"] = 0
    cf.loc["Year 1", "CFI (Capex)"] = 0  # already subtracted from cum_cash at t=0
    cf["Net_Cash_Flow"] = cf["CFO"] + cf["CFI (Capex)"]
    cf["Ending_Cash_Cum"] = df.groupby("Year")["Cum_Cash_Flow"].last().values
    cf.index = [f"Year {y}" for y in cf.index]
    return cf

print("=" * 70)
print("CASH FLOW STATEMENT — BASE CASE (Annual)")
print(f"Initial Investment: ${TOTAL_INVESTMENT:,.0f}  |  Cash starts at -${TOTAL_INVESTMENT:,.0f}")
print("=" * 70)

cf_annual = annual_cashflow(df_base)
display_cf = cf_annual.copy()
for c in cf_annual.columns:
    display_cf[c] = display_cf[c].apply(lambda x: f"${x:>12,.0f}")

display_cf.T

---
## 7. 📉 ROI & Breakeven Summary

In [ ]:
print(f"{'='*65}")
print(f"  ROI & BREAKEVEN SUMMARY  |  Total Investment: ${TOTAL_INVESTMENT:,.0f}")
print(f"{'='*65}")
print(f"{'Scenario':<14} {'M36 Revenue':>14} {'M36 Net Inc':>13} {'M36 ROI':>10} {'Cash B/E':>10}")
print(f"{'-'*65}")

for name, df in scenarios.items():
    last = df.iloc[-1]
    be = df[df["Cum_Cash_Flow"] >= 0].index.min()
    print(f"{name:<14} "
          f"${last['Revenue']:>12,.0f}  "
          f"${last['Net_Income']:>11,.0f}  "
          f"{last['ROI_%']:>8.1f}%  "
          f"Month {be if pd.notna(be) else '>36':>3}")
print(f"{'='*65}")

---
## 8. 🎲 Monte Carlo Simulation (1,000 runs)

In [ ]:
np.random.seed(42)
N_SIMS = 1_000
roi_results = []
breakevens = []

for _ in range(N_SIMS):
    # Randomly perturb volume (±25%) and price (±10%)
    vol_factor   = np.random.uniform(0.75, 1.25)
    price_factor = np.random.uniform(0.90, 1.10)
    rent_shock   = np.random.choice([0, 2000, 4000], p=[0.6, 0.3, 0.1])  # possible rent increase

    sim_services = scale_services(SERVICES, vol_factor, price_factor)
    sim_opex = {**FIXED_OPEX, "Rent (NNN)": FIXED_OPEX["Rent (NNN)"] + rent_shock}

    df_sim = build_forecast(services=sim_services, fixed_opex=sim_opex)
    roi_results.append(df_sim.iloc[-1]["ROI_%"])
    be = df_sim[df_sim["Cum_Cash_Flow"] >= 0].index.min()
    breakevens.append(be if pd.notna(be) else 37)

roi_arr = np.array(roi_results)
be_arr  = np.array(breakevens)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ROI distribution
axes[0].hist(roi_arr, bins=50, color='#1E88E5', edgecolor='white', alpha=0.85)
axes[0].axvline(np.percentile(roi_arr, 10), color='red',   linestyle='--', label='P10')
axes[0].axvline(np.percentile(roi_arr, 50), color='black', linestyle='--', label='P50')
axes[0].axvline(np.percentile(roi_arr, 90), color='green', linestyle='--', label='P90')
axes[0].set_title("Month-36 ROI Distribution (1,000 simulations)", fontweight='bold')
axes[0].set_xlabel("ROI %")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Breakeven distribution
unique, counts = np.unique(be_arr, return_counts=True)
axes[1].bar(unique, counts / N_SIMS * 100, color='#43A047', edgecolor='white', alpha=0.85)
axes[1].set_title("Cash Breakeven Month Distribution", fontweight='bold')
axes[1].set_xlabel("Month (37 = beyond 36 months)")
axes[1].set_ylabel("% of Simulations")
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))

plt.tight_layout()
plt.savefig("medispa_montecarlo.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Monte Carlo Results (n={N_SIMS:,})")
print(f"  ROI at Month 36 — P10: {np.percentile(roi_arr,10):.1f}%  "
      f"P50: {np.percentile(roi_arr,50):.1f}%  "
      f"P90: {np.percentile(roi_arr,90):.1f}%")
print(f"  Median Cash Breakeven: Month {int(np.median(be_arr))}")
print(f"  % of sims breaking even within 36 months: {(be_arr <= 36).mean()*100:.1f}%")

---
## 9. 🔬 Sensitivity Table — Volume × Price

In [ ]:
volume_factors = [0.70, 0.85, 1.00, 1.15, 1.30]
price_factors  = [0.85, 0.90, 1.00, 1.10, 1.20]

# Build ROI sensitivity table
roi_table = pd.DataFrame(index=[f"{v:.0%} vol" for v in volume_factors],
                          columns=[f"{p:.0%} price" for p in price_factors])

be_table  = roi_table.copy()

for v in volume_factors:
    for p in price_factors:
        df_s = build_forecast(services=scale_services(SERVICES, v, p))
        roi_val = df_s.iloc[-1]["ROI_%"]
        be_val  = df_s[df_s["Cum_Cash_Flow"] >= 0].index.min()
        roi_table.loc[f"{v:.0%} vol", f"{p:.0%} price"] = round(roi_val, 1)
        be_table.loc[f"{v:.0%} vol",  f"{p:.0%} price"] = int(be_val) if pd.notna(be_val) else ">36"

print("ROI % at Month 36 — Sensitivity (Volume × Price)")
print(roi_table.to_string())
print("\nCash Breakeven Month — Sensitivity (Volume × Price)")
print(be_table.to_string())

---
## 10. 💾 Export to Excel

In [ ]:
with pd.ExcelWriter("medispa_forecast.xlsx", engine="openpyxl") as writer:
    for name, df in scenarios.items():
        clean_name = name.replace("🐻 ","").replace("📊 ","").replace("🐂 ","")
        df.reset_index().to_excel(writer, sheet_name=f"{clean_name} Monthly", index=False)
        annual_pnl(df, clean_name).to_excel(writer, sheet_name=f"{clean_name} Annual P&L")
    roi_table.to_excel(writer, sheet_name="Sensitivity - ROI")
    be_table.to_excel(writer, sheet_name="Sensitivity - Breakeven")

print("✅ Exported to medispa_forecast.xlsx")